# Self-Instruct with OpenAI Batch API

This notebook builds a Python-only self-instruct dataset using the **OpenAI Batch API**. It is designed for Colab + Google Drive and uses your existing `BASE_DIR`.

Workflow:
1. Mount Drive and set API tokens
2. Write a reusable batch pipeline script into `BASE_DIR`
3. Create and submit a **task-generation batch**
4. Download and parse results into candidate tasks
5. Create and submit a **solution-generation batch**
6. Download and parse final dataset
7. Clean, convert to SFT, and upload to Hugging Face

Notes:
- Batch jobs are **asynchronous** and OpenAI documents a **24h completion window**.
- Batch output order is **not guaranteed**, so this pipeline uses `custom_id` to join outputs correctly.
- All paths are based on `BASE_DIR`.


In [ ]:
!pip install -q -U openai huggingface_hub datasets pandas orjson tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path

# Set these before running
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_KEY"
os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN"

BASE_DIR = Path("/content/drive/MyDrive/speciale/data generation/self instruct")
BASE_DIR.mkdir(parents=True, exist_ok=True)
print("BASE_DIR:", BASE_DIR)

In [ ]:
from pathlib import Path

script_path = BASE_DIR / "generate_self_instruct_openai_batch.py"
script_path.write_text('\nimport os\nimport re\nimport json\nimport time\nimport hashlib\nfrom pathlib import Path\nfrom typing import List, Dict, Any, Optional, Tuple\n\nimport orjson\nfrom openai import OpenAI\n\n# ----------------------------\n# Config\n# ----------------------------\nOPENAI_API_KEY = os.getenv("OPENAI_API_KEY")\nif not OPENAI_API_KEY:\n    raise ValueError("OPENAI_API_KEY not found in environment.")\n\nclient = OpenAI(api_key=OPENAI_API_KEY)\n\nTARGET_OBSERVATIONS = 20_000\nTASK_BATCH_SIZE = 20\nMODEL_TASKS = "gpt-4.1-mini"\nMODEL_SOLUTIONS = "gpt-4.1-mini"\n\nBASE_DIR = Path("/content/drive/MyDrive/speciale/data generation/self instruct")\nOUTPUT_DIR = BASE_DIR / "self_instruct_openai_batch_out"\nOUTPUT_DIR.mkdir(parents=True, exist_ok=True)\n\nSEED_PATH = BASE_DIR / "seed file BEST200.jsonl"\n\nTASK_REQUESTS_PATH = OUTPUT_DIR / "task_requests.jsonl"\nTASK_RESULTS_RAW_PATH = OUTPUT_DIR / "task_batch_output.jsonl"\nTASK_ERRORS_RAW_PATH = OUTPUT_DIR / "task_batch_errors.jsonl"\nTASKS_PATH = OUTPUT_DIR / "candidate_tasks.jsonl"\n\nSOLUTION_REQUESTS_PATH = OUTPUT_DIR / "solution_requests.jsonl"\nSOLUTION_RESULTS_RAW_PATH = OUTPUT_DIR / "solution_batch_output.jsonl"\nSOLUTION_ERRORS_RAW_PATH = OUTPUT_DIR / "solution_batch_errors.jsonl"\nDATASET_PATH = OUTPUT_DIR / "self_instruct_dataset.jsonl"\nREJECTS_PATH = OUTPUT_DIR / "rejected_tasks.jsonl"\n\nTASK_BATCH_INFO_PATH = OUTPUT_DIR / "task_batch_info.json"\nSOLUTION_BATCH_INFO_PATH = OUTPUT_DIR / "solution_batch_info.json"\n\nALLOWED_CATEGORIES = [\n    "strings", "lists", "dicts", "sets", "math", "recursion",\n    "sorting", "search", "dynamic_programming", "graphs",\n    "parsing", "greedy", "simulation", "combinatorics",\n    "number_theory", "geometry", "trees", "heaps", "queues", "stacks"\n]\nALLOWED_DIFFICULTIES = ["easy", "medium", "hard"]\n\nSYSTEM_TASKS = """You generate diverse Python programming tasks for synthetic instruction tuning.\nReturn ONLY valid JSON with this exact schema:\n{"tasks": [{"instruction": str, "category": str, "difficulty": str}]}\n\nConstraints:\n- Generate exactly {n_tasks} tasks.\n- Tasks must be solvable in Python.\n- Focus on algorithmic/problem-solving/data-structure style prompts.\n- No external APIs, web access, databases, GUIs, files, or package installs.\n- No duplicate or near-duplicate tasks.\n- Categories must be one of: {categories}\n- Difficulties must be one of: {difficulties}\n- Keep instructions concise but specific.\n"""\n\nSYSTEM_SOLUTIONS = """You are generating high-quality answers for synthetic instruction tuning.\nReturn ONLY valid JSON with this exact schema:\n{"output": str}\n\nConstraints:\n- Write a concise but useful assistant response that solves the task in Python.\n- Do NOT use markdown fences.\n- Do NOT mention being an AI.\n- Prefer code plus a short explanation when helpful.\n- Keep it self-contained.\n"""\n\ndef normalize_text(text: str) -> str:\n    text = text.lower().strip()\n    text = re.sub(r"\\s+", " ", text)\n    return text\n\ndef fingerprint(text: str) -> str:\n    return hashlib.sha256(normalize_text(text).encode("utf-8")).hexdigest()\n\ndef load_jsonl(path: Path) -> List[Dict[str, Any]]:\n    rows = []\n    with path.open("r", encoding="utf-8") as f:\n        for line in f:\n            if line.strip():\n                rows.append(json.loads(line))\n    return rows\n\ndef write_jsonl(path: Path, rows: List[Dict[str, Any]]) -> None:\n    with path.open("w", encoding="utf-8") as f:\n        for row in rows:\n            f.write(json.dumps(row, ensure_ascii=False) + "\\n")\n\ndef append_jsonl(path: Path, rows: List[Dict[str, Any]]) -> None:\n    with path.open("a", encoding="utf-8") as f:\n        for row in rows:\n            f.write(json.dumps(row, ensure_ascii=False) + "\\n")\n\ndef load_seed_examples(seed_path: Path, max_examples: int = 12) -> List[str]:\n    rows = load_jsonl(seed_path)\n    examples = []\n    for row in rows[:max_examples]:\n        text = row.get("instruction") or row.get("prompt") or row.get("task") or ""\n        text = text.strip()\n        if text:\n            examples.append(text)\n    if not examples:\n        raise ValueError(f"No usable seed tasks found in {seed_path}")\n    return examples\n\ndef make_task_user_prompt(seed_examples: List[str], n_tasks: int) -> str:\n    formatted = "\\n".join(f"- {x}" for x in seed_examples)\n    return f"""Using the style diversity of these seed tasks, generate {n_tasks} NEW Python programming instructions.\n\nSeed examples:\n{formatted}\n\nReturn only JSON.\n"""\n\ndef safe_json_loads(text: str) -> Any:\n    try:\n        return json.loads(text)\n    except Exception:\n        return orjson.loads(text)\n\ndef response_text_from_batch_line(obj: Dict[str, Any]) -> str:\n    body = obj["response"]["body"]\n    # Responses API output_text\n    if "output_text" in body and body["output_text"]:\n        return body["output_text"]\n\n    # Fallback: parse output list\n    output = body.get("output", [])\n    parts = []\n    for item in output:\n        for content in item.get("content", []):\n            if content.get("type") == "output_text" and content.get("text"):\n                parts.append(content["text"])\n    return "\\n".join(parts).strip()\n\ndef save_batch_info(path: Path, batch: Any) -> None:\n    data = {\n        "id": batch.id,\n        "status": batch.status,\n        "input_file_id": getattr(batch, "input_file_id", None),\n        "output_file_id": getattr(batch, "output_file_id", None),\n        "error_file_id": getattr(batch, "error_file_id", None),\n        "endpoint": getattr(batch, "endpoint", None),\n        "completion_window": getattr(batch, "completion_window", None),\n    }\n    path.write_text(json.dumps(data, indent=2), encoding="utf-8")\n\ndef load_batch_info(path: Path) -> Dict[str, Any]:\n    return json.loads(path.read_text(encoding="utf-8"))\n\ndef create_batch_input_file(requests_path: Path) -> str:\n    uploaded = client.files.create(file=requests_path.open("rb"), purpose="batch")\n    return uploaded.id\n\ndef create_batch(file_id: str, metadata: Optional[Dict[str, str]] = None):\n    batch = client.batches.create(\n        input_file_id=file_id,\n        endpoint="/v1/responses",\n        completion_window="24h",\n        metadata=metadata or {},\n    )\n    return batch\n\ndef retrieve_batch(batch_id: str):\n    return client.batches.retrieve(batch_id)\n\ndef download_file(file_id: str, out_path: Path) -> None:\n    content = client.files.content(file_id)\n    text = content.text\n    out_path.write_text(text, encoding="utf-8")\n\ndef build_task_requests(target_observations: int = TARGET_OBSERVATIONS, task_batch_size: int = TASK_BATCH_SIZE) -> int:\n    seed_examples = load_seed_examples(SEED_PATH, max_examples=12)\n    n_calls = (target_observations + task_batch_size - 1) // task_batch_size\n\n    with TASK_REQUESTS_PATH.open("w", encoding="utf-8") as f:\n        for i in range(n_calls):\n            body = {\n                "model": MODEL_TASKS,\n                "input": [\n                    {\n                        "role": "system",\n                        "content": SYSTEM_TASKS.format(\n                            n_tasks=task_batch_size,\n                            categories=", ".join(ALLOWED_CATEGORIES),\n                            difficulties=", ".join(ALLOWED_DIFFICULTIES),\n                        ),\n                    },\n                    {\n                        "role": "user",\n                        "content": make_task_user_prompt(seed_examples, task_batch_size),\n                    },\n                ],\n            }\n            req = {\n                "custom_id": f"task_gen_{i:05d}",\n                "method": "POST",\n                "url": "/v1/responses",\n                "body": body,\n            }\n            f.write(json.dumps(req, ensure_ascii=False) + "\\n")\n\n    return n_calls\n\ndef submit_task_batch() -> Dict[str, Any]:\n    file_id = create_batch_input_file(TASK_REQUESTS_PATH)\n    batch = create_batch(file_id, metadata={"stage": "task_generation"})\n    save_batch_info(TASK_BATCH_INFO_PATH, batch)\n    return load_batch_info(TASK_BATCH_INFO_PATH)\n\ndef download_task_batch_outputs() -> Dict[str, Any]:\n    info = load_batch_info(TASK_BATCH_INFO_PATH)\n    batch = retrieve_batch(info["id"])\n    save_batch_info(TASK_BATCH_INFO_PATH, batch)\n    refreshed = load_batch_info(TASK_BATCH_INFO_PATH)\n\n    if refreshed["status"] != "completed":\n        raise RuntimeError(f"Task batch not completed yet. Current status: {refreshed[\'status\']}")\n\n    if refreshed.get("output_file_id"):\n        download_file(refreshed["output_file_id"], TASK_RESULTS_RAW_PATH)\n    if refreshed.get("error_file_id"):\n        download_file(refreshed["error_file_id"], TASK_ERRORS_RAW_PATH)\n\n    return refreshed\n\ndef validate_task(task: Dict[str, Any]) -> Tuple[bool, str]:\n    instruction = (task.get("instruction") or "").strip()\n    category = (task.get("category") or "").strip()\n    difficulty = (task.get("difficulty") or "").strip()\n\n    if not instruction:\n        return False, "empty_instruction"\n    if category not in ALLOWED_CATEGORIES:\n        return False, "invalid_category"\n    if difficulty not in ALLOWED_DIFFICULTIES:\n        return False, "invalid_difficulty"\n\n    lowered = instruction.lower()\n    banned = ["browser", "internet", "api", "sql server", "mongodb", "postgres", "http request", "selenium"]\n    if any(x in lowered for x in banned):\n        return False, "out_of_scope"\n\n    return True, "ok"\n\ndef parse_task_batch_output() -> Dict[str, int]:\n    if not TASK_RESULTS_RAW_PATH.exists():\n        raise FileNotFoundError(f"Missing {TASK_RESULTS_RAW_PATH}")\n\n    accepted = []\n    rejected = []\n    seen = set()\n\n    with TASK_RESULTS_RAW_PATH.open("r", encoding="utf-8") as f:\n        for line in f:\n            if not line.strip():\n                continue\n            obj = json.loads(line)\n            text = response_text_from_batch_line(obj)\n            try:\n                payload = safe_json_loads(text)\n            except Exception:\n                rejected.append({"custom_id": obj.get("custom_id"), "reason": "invalid_json", "raw": text[:2000]})\n                continue\n\n            tasks = payload.get("tasks", [])\n            if not isinstance(tasks, list):\n                rejected.append({"custom_id": obj.get("custom_id"), "reason": "missing_tasks_list", "raw": text[:2000]})\n                continue\n\n            for t in tasks:\n                ok, reason = validate_task(t)\n                if not ok:\n                    rejected.append({"custom_id": obj.get("custom_id"), "reason": reason, "task": t})\n                    continue\n\n                fp = fingerprint(t["instruction"])\n                if fp in seen:\n                    rejected.append({"custom_id": obj.get("custom_id"), "reason": "duplicate", "task": t})\n                    continue\n                seen.add(fp)\n\n                accepted.append({\n                    "id": fp[:16],\n                    "instruction": t["instruction"].strip(),\n                    "category": t["category"].strip(),\n                    "difficulty": t["difficulty"].strip(),\n                    "language": "python",\n                    "source_method": "self_instruct_batch_openai",\n                })\n\n    if len(accepted) > TARGET_OBSERVATIONS:\n        accepted = accepted[:TARGET_OBSERVATIONS]\n\n    write_jsonl(TASKS_PATH, accepted)\n    write_jsonl(REJECTS_PATH, rejected)\n\n    return {\n        "accepted_tasks": len(accepted),\n        "rejected_items": len(rejected),\n    }\n\ndef build_solution_requests(limit: Optional[int] = None) -> int:\n    tasks = load_jsonl(TASKS_PATH)\n    if limit is not None:\n        tasks = tasks[:limit]\n\n    with SOLUTION_REQUESTS_PATH.open("w", encoding="utf-8") as f:\n        for idx, task in enumerate(tasks):\n            body = {\n                "model": MODEL_SOLUTIONS,\n                "input": [\n                    {"role": "system", "content": SYSTEM_SOLUTIONS},\n                    {"role": "user", "content": task["instruction"]},\n                ],\n            }\n            req = {\n                "custom_id": f"solve::{task[\'id\']}",\n                "method": "POST",\n                "url": "/v1/responses",\n                "body": body,\n            }\n            f.write(json.dumps(req, ensure_ascii=False) + "\\n")\n\n    return len(tasks)\n\ndef submit_solution_batch() -> Dict[str, Any]:\n    file_id = create_batch_input_file(SOLUTION_REQUESTS_PATH)\n    batch = create_batch(file_id, metadata={"stage": "solution_generation"})\n    save_batch_info(SOLUTION_BATCH_INFO_PATH, batch)\n    return load_batch_info(SOLUTION_BATCH_INFO_PATH)\n\ndef download_solution_batch_outputs() -> Dict[str, Any]:\n    info = load_batch_info(SOLUTION_BATCH_INFO_PATH)\n    batch = retrieve_batch(info["id"])\n    save_batch_info(SOLUTION_BATCH_INFO_PATH, batch)\n    refreshed = load_batch_info(SOLUTION_BATCH_INFO_PATH)\n\n    if refreshed["status"] != "completed":\n        raise RuntimeError(f"Solution batch not completed yet. Current status: {refreshed[\'status\']}")\n\n    if refreshed.get("output_file_id"):\n        download_file(refreshed["output_file_id"], SOLUTION_RESULTS_RAW_PATH)\n    if refreshed.get("error_file_id"):\n        download_file(refreshed["error_file_id"], SOLUTION_ERRORS_RAW_PATH)\n\n    return refreshed\n\ndef parse_solution_batch_output() -> Dict[str, int]:\n    tasks = {row["id"]: row for row in load_jsonl(TASKS_PATH)}\n    dataset = []\n    failed = 0\n\n    with SOLUTION_RESULTS_RAW_PATH.open("r", encoding="utf-8") as f:\n        for line in f:\n            if not line.strip():\n                continue\n            obj = json.loads(line)\n            custom_id = obj.get("custom_id", "")\n            if not custom_id.startswith("solve::"):\n                failed += 1\n                continue\n            task_id = custom_id.split("solve::", 1)[1]\n            task = tasks.get(task_id)\n            if task is None:\n                failed += 1\n                continue\n\n            text = response_text_from_batch_line(obj)\n            try:\n                payload = safe_json_loads(text)\n                output = payload.get("output", "").strip()\n            except Exception:\n                output = text.strip()\n\n            if not output:\n                failed += 1\n                continue\n\n            dataset.append({\n                **task,\n                "output": output,\n            })\n\n    write_jsonl(DATASET_PATH, dataset)\n\n    return {\n        "dataset_rows": len(dataset),\n        "failed_or_missing": failed,\n    }\n\ndef batch_status(which: str = "task") -> Dict[str, Any]:\n    info_path = TASK_BATCH_INFO_PATH if which == "task" else SOLUTION_BATCH_INFO_PATH\n    info = load_batch_info(info_path)\n    batch = retrieve_batch(info["id"])\n    save_batch_info(info_path, batch)\n    return load_batch_info(info_path)\n\ndef clean_dataset() -> Dict[str, Any]:\n    input_path = DATASET_PATH\n    output_path = OUTPUT_DIR / "self_instruct_dataset_clean.jsonl"\n\n    rows = load_jsonl(input_path)\n    seen = set()\n    clean = []\n\n    for row in rows:\n        instruction = (row.get("instruction") or "").strip()\n        output = (row.get("output") or "").strip()\n        if not instruction or not output:\n            continue\n        if row.get("language") != "python":\n            continue\n        if "```" in output:\n            continue\n        if len(output) < 20:\n            continue\n\n        fp = fingerprint(instruction)\n        if fp in seen:\n            continue\n        seen.add(fp)\n        clean.append(row)\n\n    write_jsonl(output_path, clean)\n    return {"clean_rows": len(clean), "path": str(output_path)}\n\ndef make_sft() -> Dict[str, Any]:\n    src = OUTPUT_DIR / "self_instruct_dataset_clean.jsonl"\n    dst = OUTPUT_DIR / "self_instruct_sft.jsonl"\n\n    with src.open("r", encoding="utf-8") as fin, dst.open("w", encoding="utf-8") as fout:\n        for line in fin:\n            row = json.loads(line)\n            sft_row = {\n                "messages": [\n                    {"role": "user", "content": row["instruction"]},\n                    {"role": "assistant", "content": row["output"]},\n                ],\n                "metadata": {\n                    "id": row["id"],\n                    "category": row["category"],\n                    "difficulty": row["difficulty"],\n                    "source_method": row["source_method"],\n                },\n            }\n            fout.write(json.dumps(sft_row, ensure_ascii=False) + "\\n")\n\n    return {"sft_path": str(dst)}\n\ndef push_clean_dataset_to_hub(repo_id: str, private: bool = False) -> None:\n    hf_token = os.getenv("HF_TOKEN")\n    if not hf_token:\n        raise ValueError("HF_TOKEN not found in environment.")\n\n    from huggingface_hub import login\n    from datasets import Dataset\n\n    login(token=hf_token)\n\n    rows = load_jsonl(OUTPUT_DIR / "self_instruct_dataset_clean.jsonl")\n    dataset = Dataset.from_list(rows)\n    dataset.push_to_hub(repo_id, private=private)\n', encoding="utf-8")
print("Wrote:", script_path)

In [ ]:
# Import the pipeline script
import sys
sys.path.append(str(BASE_DIR))

import generate_self_instruct_openai_batch as sib

print("Output dir:", sib.OUTPUT_DIR)
print("Seed path:", sib.SEED_PATH)

In [ ]:
# Step 1: Build task-generation requests
n_task_calls = sib.build_task_requests()
print("Task-generation requests written:", n_task_calls)
print("Batch request file:", sib.TASK_REQUESTS_PATH)

In [ ]:
# Step 2: Submit the task-generation batch
task_batch = sib.submit_task_batch()
task_batch

In [ ]:
# Step 3: Check task batch status
sib.batch_status("task")

In [ ]:
# Step 4: When the task batch is completed, download outputs
sib.download_task_batch_outputs()

In [ ]:
# Step 5: Parse task outputs into candidate_tasks.jsonl
sib.parse_task_batch_output()

In [ ]:
# Optional: inspect accepted task count
from pathlib import Path
accepted_count = sum(1 for _ in open(sib.TASKS_PATH, "r", encoding="utf-8"))
rejected_count = sum(1 for _ in open(sib.REJECTS_PATH, "r", encoding="utf-8"))
print("Accepted:", accepted_count)
print("Rejected:", rejected_count)

In [ ]:
# Step 6: Build one solution-generation request per accepted task
n_solution_calls = sib.build_solution_requests()
print("Solution-generation requests written:", n_solution_calls)
print("Batch request file:", sib.SOLUTION_REQUESTS_PATH)

In [ ]:
# Step 7: Submit the solution-generation batch
solution_batch = sib.submit_solution_batch()
solution_batch

In [ ]:
# Step 8: Check solution batch status
sib.batch_status("solution")

In [ ]:
# Step 9: When the solution batch is completed, download outputs
sib.download_solution_batch_outputs()

In [ ]:
# Step 10: Parse solution outputs into the final dataset
sib.parse_solution_batch_output()

In [ ]:
# Step 11: Clean and deduplicate
sib.clean_dataset()

In [ ]:
# Step 12: Convert to SFT format
sib.make_sft()

In [ ]:
# Step 13: Push cleaned dataset to Hugging Face
REPO_ID = "your-username/self-instruct-python-20k-batch"
PRIVATE = False

# Uncomment to push:
# sib.push_clean_dataset_to_hub(REPO_ID, private=PRIVATE)

print("Ready to push:", REPO_ID)

## Practical notes

- For **20,000 observations**, task generation creates about **1,000 tasks per 50 calls?** No — in this notebook it creates **20 tasks per call**, so it will write about **1,000 task-generation requests**? Actually: `20,000 / 20 = 1,000`.  
- Then it creates roughly **one solution request per accepted task**, so around **20,000 solution requests**.
- With the Batch API, the wall-clock time is dominated by OpenAI's asynchronous processing window rather than notebook runtime. The notebook itself should run quickly for each step, while each batch can complete any time within the documented **24h** window.

You can safely disconnect and return later:
- the batch IDs are stored in `task_batch_info.json` and `solution_batch_info.json`
- rerun the import cell and then the status/download/parse cells
